# SafeQuery on Colab (Run 9): the reported score next to the raw behaviour, with settings saved in every results file
**Lexsi Labs hackathon, Agentic Fine-tuning track**: AgentTune + AuditKIT on **Tiny Aya Global** (Cohere Labs). No other model is used.

**Why this run.** Organizer feedback, in two rounds:
1. The stop-token / `trim_turn` configuration is part of the reported system identity (answered by the README *serving contract*).
2. *"Reporting raw overrun alongside the trimmed score would make the wrapper's contribution inspectable without discarding its practical value. Keeping those settings with each results file also seems useful: future readers could tell whether a changed score came from the adapter, decoding, or post-processing."*

**What it does. No training.** It reuses the adapters Run 6 saved to Google Drive (or the published Hub adapters).
1. Evaluates base, SFT and GRPO on the 111 held-out tasks under two serving wrappers:
   - `safequery`: the reported settings (greedy, stop on every end-of-turn token and on `"Tool result for"`, then `trim_turn()`).
   - `plain`: a bare `generate()`: the model's default stopping, no stop string, no trimming.
2. **Raw overrun next to the trimmed score.** Every turn is logged before trimming, with why it stopped (`model_eos`, `wrapper_eos`, `stop_string`, `token_limit`) and whether trimming changed how it parsed. Each `metrics.json` gets a `wrapper_report` with:
   - the reported (trimmed) task success;
   - task success with **no wrapper intervention**;
   - the number of successes that needed the wrapper;
   - the raw overrun rate.
3. **Settings in every results file.** `metrics.json` and `benchmarks.json` carry a `serving` block, grouped as `adapter` / `decoding` / `prompt` / `post_processing` / `versions`. It holds a `serving_id` and one hash per group. A `provenance` block records the code commit, adapter weight hash, adapter source and test-set hash. `scripts/explain_diff.py A B` then names which group changed between two results files.
4. Writes `results_run9/WRAPPERS.md`, including a check that the `safequery` rows still equal Run 6.

**GPU: A100** (as in Run 6), so the check against Run 6 compares like with like. Install 5-15 min, each evaluation ~25-40 min (plain turns can run to the token limit), 6 evaluations ≈ 3-4 h. Set `RUN_GRPO = False` to halve it.
Results are saved to **`MyDrive/safequery_run9`** after every evaluation, and every step skips work that's already done.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import sys, torch; print(sys.version)  # AgentTune needs Python >= 3.12
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("GPU:", gpu)
if "A100" not in gpu:
    print("WARNING: Run 6 ran on an A100. On another GPU the scores can differ slightly from Run 6 for numeric reasons alone.")

## 0 - Get the Run 9 code from GitHub
Push the `LLMHackathon2026_v9` folder to the `run9` branch first (`main` stays unchanged until the results are in).

In [ ]:
import os
PROJ = "/content/LLMHackathon2026"
REPO_URL = "https://github.com/siva2k16/LLMHackathon2026"
REPO_BRANCH = "run9"
if not os.path.exists(f"{PROJ}/scripts/build_data.py"):
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {PROJ}
else:
    !git -C {PROJ} fetch -q origin {REPO_BRANCH} && git -C {PROJ} checkout -q {REPO_BRANCH} && git -C {PROJ} pull -q origin {REPO_BRANCH}
%cd {PROJ}
!git log --oneline -1
ok = all(os.path.exists(p) for p in ["safequery/serving.py", "scripts/serving_report.py", "scripts/explain_diff.py"])
print("Run 9 code present:", ok)
assert ok, f"Push the v9 code to the {REPO_BRANCH} branch on GitHub first"

## 1 - Install AgentTune and AuditKIT (unmodified, from Lexsi Labs)
5-15 minutes. Red **dependency conflict** warnings are harmless. Colab's preinstalled `torchao` is removed, because it breaks the newer `peft` (the error seen in the demo notebook).
**After it installs, use Runtime -> Restart session**, then continue from the next cell.

In [ ]:
!git clone -q https://github.com/Lexsi-Labs/AgentTune.git /content/AgentTune 2>/dev/null || echo "AgentTune already cloned"
!git clone -q https://github.com/Lexsi-Labs/AuditKIT.git /content/AuditKIT 2>/dev/null || echo "AuditKIT already cloned"
import subprocess
if subprocess.run(["python", "-c", "import agenttune, auditkit, peft"], capture_output=True).returncode == 0:
    print("AgentTune + AuditKIT already installed - skipping")
else:
    !pip install -q --only-binary=pandas -e /content/AgentTune -e /content/AuditKIT
    !pip uninstall -q -y torchao
    print("Installed. Now: Runtime -> Restart session, then run the next cells.")

In [ ]:
import sys
for p in ["/content/AgentTune/src", "/content/AuditKIT/src", "/content/AuditKIT"]:
    if p not in sys.path: sys.path.insert(0, p)
import torch, transformers, peft, agenttune, auditkit
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| transformers", transformers.__version__, "| peft", peft.__version__)

## 1b - Hugging Face login
Tiny Aya Global is **gated**: accept the terms at https://huggingface.co/CohereLabs/tiny-aya-global first.
It uses the Colab secret `HF_TOKEN` if you have set one, otherwise it shows a login box. A **read** token is enough, unless you want step 9 (upload) as well. Never paste tokens into code cells.

In [ ]:
from huggingface_hub import login, notebook_login, model_info
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN")); print("logged in with the HF_TOKEN secret")
except Exception:
    notebook_login()

In [ ]:
from huggingface_hub import model_info
print("Tiny Aya access OK:", model_info("CohereLabs/tiny-aya-global").id)

## Config - **re-run this cell after every restart or disconnect**

In [ ]:
import os, glob, shutil, hashlib, json, time, torch
PROJ = "/content/LLMHackathon2026"
%cd {PROJ}

MODEL     = "CohereLabs/tiny-aya-global"   # the only model used (Cohere Labs, as the rules require)
RES       = "results_run9"
RUN_GRPO  = True     # False = base + SFT only (about half the time)
RUN_BENCH = False    # True = also BFCL + ARC under both wrappers (+ ~1.5 h)
LIMIT     = None     # None = all 111 test tasks
LOAD_4BIT = False

# Where the adapters come from. "drive_run6" reuses what Run 6 saved to Drive (default);
# "drive_run5" uses the Run 5 Drive folder; "hub" downloads the published adapters.
ADAPTER_SOURCE = "drive_run6"
DRIVE_RUNS = {"drive_run6": "/content/drive/MyDrive/safequery_run6/runs",
              "drive_run5": "/content/drive/MyDrive/safequery/runs"}
BASELINE   = {"drive_run6": "results_run6", "drive_run5": "results", "hub": "results_run6"}[ADAPTER_SOURCE]
HF_ADAPTERS = {"sft":  "Siva2022/safequery-tiny-aya-global-sft-lora",
               "grpo": "Siva2022/safequery-tiny-aya-global-grpo-lora"}
VERIFY_AGAINST_HUB = True   # also download the Hub adapters and check they are the same weights

LIMIT_ARG = f"--limit {LIMIT}" if LIMIT else ""
Q_ARG     = "--load-4bit" if LOAD_4BIT else ""
DRIVE_OUT = "/content/drive/MyDrive/safequery_run9"
GPU_NAME  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
MODELS    = ["base", "sft"] + (["grpo"] if RUN_GRPO else [])
os.makedirs(RES, exist_ok=True)

RUN5_MD5 = {"tasks_train.jsonl": "1297c43b4683d016f9529620a41bb0b9", "tasks_test.jsonl": "aa5cd9fac7e2f137f52974e1d6a0643a",
            "sft_train.jsonl": "ab308961f042e5cefe54205c1eef60df", "grpo_train.jsonl": "e38fa26dcbf4e7c5a8815d4c9eaf99c0"}

def _mount():
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

def _md5(p):
    return hashlib.md5(open(p, "rb").read()).hexdigest() if os.path.exists(p) else None

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()[:12]

def _done(name, fname="metrics.json"):
    p = f"{RES}/{name}/{fname}"
    return os.path.exists(p) and "serving_id" in json.dumps(json.load(open(p)).get("serving", {}))

def run_name(model_name, wrapper):
    return model_name if wrapper == "safequery" else f"{model_name}_plain"

def save_to_drive():
    _mount(); os.makedirs(DRIVE_OUT, exist_ok=True)
    shutil.copytree(RES, f"{DRIVE_OUT}/{RES}", dirs_exist_ok=True)
    print("saved ->", DRIVE_OUT)

TIMES = {}
def timed(label, t0):
    TIMES[label] = round((time.time() - t0) / 60, 1); print(f"[time] {label}: {TIMES[label]} min")

print("GPU:", GPU_NAME, "| adapters from", ADAPTER_SOURCE, "| baseline", BASELINE, "| cwd", os.getcwd())

## R - Status, and restore after a disconnect
Set `RESTORE = True` after a disconnect: it copies finished evaluations back from `MyDrive/safequery_run9`, so they are skipped.

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

RESTORE = False
if RESTORE:
    _mount()
    if os.path.exists(f"{DRIVE_OUT}/{RES}"):
        shutil.copytree(f"{DRIVE_OUT}/{RES}", RES, dirs_exist_ok=True)
for m in MODELS:
    for w in ("safequery", "plain"):
        n = run_name(m, w)
        print(f"{n:12s} eval: {'done' if _done(n) else '-':5s}  bench: {'done' if _done(n, 'benchmarks.json') else '-'}")
print("adapters:", sorted(os.listdir("adapters")) if os.path.isdir("adapters") else "not fetched yet")

## 2 - Build the database and tasks, check they match Run 5/6

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

%cd {PROJ}
!python scripts/build_data.py
ok = True
for f, want in RUN5_MD5.items():
    got = _md5(f"data/{f}")
    print(f"{f:20s}", "MATCH" if got == want else f"DIFFERENT ({got})")
    ok &= got == want
print("all match:", ok)

## 3 - Reuse the saved adapters (Drive), check them against the published ones (Hub)
It copies `runs/sft` and `runs/grpo` from the Run 6 Drive folder into `adapters/`. If Drive doesn't have them, it falls back to the Hub.
The weight hashes and the source go to `results_run9/adapters.json` and into every results file's `provenance`.

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

%cd {PROJ}
from huggingface_hub import HfApi, snapshot_download
api = HfApi(); ADAPTER_INFO = {}
for name in (["sft", "grpo"] if RUN_GRPO else ["sft"]):
    dst = f"adapters/{name}"
    src = None
    if ADAPTER_SOURCE.startswith("drive"):
        _mount()
        d = f"{DRIVE_RUNS[ADAPTER_SOURCE]}/{name}"
        if os.path.exists(f"{d}/adapter_model.safetensors"):
            shutil.copytree(d, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
            src = f"drive:{d.replace('/content/drive/MyDrive/', '')}"
        else:
            print(f"{name}: not on Drive at {d} - using the Hub instead")
    if src is None:
        snapshot_download(HF_ADAPTERS[name], local_dir=dst)
        src = f"hub:{HF_ADAPTERS[name]}@{api.model_info(HF_ADAPTERS[name]).sha[:10]}"
    info = {"source": src, "local": dst, "weights_sha256": _sha(f"{dst}/adapter_model.safetensors")}
    if VERIFY_AGAINST_HUB:
        hub_dir = snapshot_download(HF_ADAPTERS[name], local_dir=f"adapters_hub/{name}")
        hub_sha = _sha(f"{hub_dir}/adapter_model.safetensors")
        info.update(hub_repo=HF_ADAPTERS[name], hub_commit=api.model_info(HF_ADAPTERS[name]).sha,
                    hub_weights_sha256=hub_sha, same_as_hub=hub_sha == info["weights_sha256"])
    ADAPTER_INFO[name] = info
    print(name, "|", src, "| weights", info["weights_sha256"], "| same as Hub:", info.get("same_as_hub", "not checked"))
json.dump(ADAPTER_INFO, open(f"{RES}/adapters.json", "w"), indent=2)

## 4 - Merge the SFT adapter (GRPO's starting model)
The GRPO adapter was trained on top of the merged SFT model, so it is evaluated on that base.

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

%cd {PROJ}
if RUN_GRPO and not os.path.exists("runs/sft_merged/config.json"):
    t0 = time.time()
    !python scripts/merge_adapter.py --model {MODEL} --adapter adapters/sft --out runs/sft_merged
    timed("merge", t0)
else:
    print("merge not needed or already done")

## 5 - Evaluate every model under both wrappers
`safequery` = the reported settings · `plain` = bare `generate()`. Saved to Drive after each evaluation.

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

%cd {PROJ}
ADAPTER_INFO = json.load(open(f"{RES}/adapters.json"))
SPEC = {"base": (MODEL, None), "sft": (MODEL, "adapters/sft"), "grpo": ("runs/sft_merged", "adapters/grpo")}
for m in MODELS:
    model, adapter = SPEC[m]
    for wrapper in ("safequery", "plain"):
        name = run_name(m, wrapper)
        if _done(name):
            print(name, "already done - skipping"); continue
        ad = f"--adapter {adapter} --adapter-source {ADAPTER_INFO[m]['source']}" if adapter else ""
        t0 = time.time()
        !python scripts/eval_model.py --name {name} --model {model} {ad} --wrapper {wrapper} --results {RES} {LIMIT_ARG} {Q_ARG}
        timed(f"{name} eval", t0)
        save_to_drive()

## 6 - *Optional*: BFCL + ARC under both wrappers (`RUN_BENCH = True`)
BFCL and ARC go through the same wrapper, so they get the same `serving` block and raw-turn log.

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

%cd {PROJ}
if RUN_BENCH:
    ADAPTER_INFO = json.load(open(f"{RES}/adapters.json"))
    SPEC = {"base": (MODEL, None), "sft": (MODEL, "adapters/sft"), "grpo": ("runs/sft_merged", "adapters/grpo")}
    for m in MODELS:
        model, adapter = SPEC[m]
        for wrapper in ("safequery", "plain"):
            name = run_name(m, wrapper)
            if _done(name, "benchmarks.json"):
                print(name, "benchmarks already done - skipping"); continue
            ad = f"--adapter {adapter} --adapter-source {ADAPTER_INFO[m]['source']}" if adapter else ""
            t0 = time.time()
            !python scripts/eval_benchmarks.py --name {name} --model {model} {ad} --wrapper {wrapper} --results {RES} {Q_ARG}
            timed(f"{name} bench", t0)
            save_to_drive()
else:
    print("RUN_BENCH is False - skipping")

## 7 - Report: the trimmed score next to the raw behaviour, and what changed between results
- `WRAPPERS.md`: per model and wrapper, the reported (trimmed) score, the score with no wrapper intervention, successes that needed the wrapper, raw overrun and stop reasons, plus the `serving_id`; what changed between the two wrappers; and a check against Run 6.
- `COMPARISON.md`: the usual before/after table, now with the raw-behaviour rows and `serving_id`.
- `DIFF_run6_vs_run9_sft.md`: what `explain_diff.py` says about an older result. Run 6 files have no serving block, so it reports "not recorded". That is the gap this run closes.

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

%cd {PROJ}
from IPython.display import Markdown, display
!python scripts/serving_report.py --results {RES} --models {' '.join(MODELS)} --baseline {BASELINE} --out {RES}/WRAPPERS.md > /dev/null
runs = " ".join(run_name(m, w) for m in MODELS for w in ("safequery", "plain") if os.path.exists(f"{RES}/{run_name(m, w)}/metrics.json"))
!python scripts/compare.py --results {RES} --runs {runs} --out {RES}/COMPARISON.md > /dev/null
!python scripts/explain_diff.py {BASELINE}/sft {RES}/sft --out {RES}/DIFF_{BASELINE}_vs_run9_sft.md > /dev/null
display(Markdown(open(f"{RES}/WRAPPERS.md", encoding="utf-8").read()))

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

# One raw turn where the model ran past its own turn, before trimming, as an example
%cd {PROJ}
for name in ["sft_plain", "grpo_plain", "sft", "grpo", "base_plain"]:
    p = f"{RES}/{name}/raw_turns.jsonl"
    if os.path.exists(p):
        ex = next((r for r in map(json.loads, open(p, encoding="utf-8")) if r["overrun"]), None)
        if ex:
            print(f"--- {name} · {ex['task_id']} · stop: {ex['stop_reason']} · {ex['new_tokens']} new tokens ---")
            print("RAW:\n", ex["raw"][:900]); print("\nRETURNED (what the agent loop saw):\n", ex["returned"][:400]); break
else:
    print("no overrun found")

## 8 - Save and download

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

%cd {PROJ}
json.dump({"gpu": GPU_NAME, "adapter_source": ADAPTER_SOURCE, "minutes": TIMES}, open(f"{RES}/run9_timing.json", "w"), indent=2)
save_to_drive()
!zip -r -q /content/safequery_results_run9.zip {RES}
!ls -lh /content/safequery_results_run9.zip
from google.colab import files
files.download("/content/safequery_results_run9.zip")

## 9 - *Optional*: publish the Run 9 logs to the public HF dataset
The rules ask for logs to be public on Hugging Face. This uploads `results_run9/` (metrics, raw turns, serving settings) to `run9/` in the eval-traces dataset. It needs a **write** token. Leave `UPLOAD = False` until you have checked the report.

In [ ]:
if "RES" not in globals(): raise RuntimeError("Session restarted: re-run the Config cell first")

UPLOAD = False
DATASET = "Siva2022/safequery-eval-traces"
if UPLOAD:
    from huggingface_hub import HfApi
    HfApi().upload_folder(folder_path=RES, path_in_repo="run9", repo_id=DATASET, repo_type="dataset",
                          commit_message="Run 9: serving settings in every results file, raw overrun next to trimmed score")
    print("uploaded ->", f"https://huggingface.co/datasets/{DATASET}/tree/main/run9")
else:
    print("UPLOAD is False - nothing uploaded")

## If Colab disconnects after the evaluations
The report is pure Python, so a **CPU** runtime is enough. Run step 0, then Config, then R with `RESTORE = True`, then step 7 and step 8. No install, login or GPU is needed for those.